# IndexTTS-2.5 Colab 部署

在 Colab 免费 GPU 上运行 **IndexTTS-2.5**（支持中/英/日/西/阿多语言 + 语音克隆 + 情感控制）。

运行前请确认：`运行时 → 更改运行时类型 → 硬件加速器 = GPU`（T4 即可）。

> 若 HuggingFace 下载慢，可在下方「下载模型」步骤前运行 `%env HF_ENDPOINT=https://hf-mirror.com`。

In [ ]:
# 1. 检查 GPU
!nvidia-smi

In [ ]:
# 2. 克隆仓库（默认用上游 index-tts/index-tts；如需 fork 改为你的地址）
REPO_URL = "https://github.com/index-tts/index-tts.git"
!git clone --depth 1 $REPO_URL index-tts
%cd index-tts

In [ ]:
# 3. 安装 uv 并同步依赖（含 webui / deepspeed 等全部 extras）
!pip install -U uv -q
!uv sync --all-extras
# 国内镜像加速（可选）:
# !uv sync --all-extras --default-index "https://mirrors.aliyun.com/pypi/simple"

In [ ]:
# 4. 下载 IndexTTS-2.5 权重到 checkpoints/（约 3~4 GB）
!uv tool install huggingface-hub -q
!hf download IndexTeam/IndexTTS-2.5 --local-dir=checkpoints
# 备选：ModelScope
# !uv tool install modelscope -q
# !modelscope download --model IndexTeam/IndexTTS-2.5 --local_dir checkpoints

In [ ]:
# 5. 环境自检
!uv run tools/gpu_check.py

In [ ]:
# 6. 初始化 IndexTTS-2.5（BF16 省显存；示例音频首次运行自动下载）
from indextts.utils.examples_downloader import ensure_examples_available
ensure_examples_available()

from indextts.infer_v2_5 import IndexTTS2
tts = IndexTTS2(cfg_path="checkpoints/config.yaml", model_dir="checkpoints", use_bf16=True)

In [ ]:
# 7. 语音克隆（单参考音频 + 指定语言）
from IPython.display import Audio

tts.infer(
    spk_audio_prompt="examples/voice_01.wav",
    text="你好，我是 IndexTTS-2.5，欢迎测试多语言语音合成。",
    lang="ZH",
    output_path="output_zh.wav",
    verbose=True,
)
Audio("output_zh.wav")

In [ ]:
# 8. 情感控制（情感参考音频 + emo_alpha 强度）
tts.infer(
    spk_audio_prompt="examples/voice_07.wav",
    text="酒楼丧尽天良，开始借机竞拍房间，哎，一群蠢货。",
    lang="ZH",
    output_path="output_emo.wav",
    emo_audio_prompt="examples/emo_sad.wav",
    emo_alpha=0.9,
    verbose=True,
)
Audio("output_emo.wav")

In [ ]:
# 9. 下载生成结果（保存到本地）
from google.colab import files
files.download("output_zh.wav")
files.download("output_emo.wav")

## （可选）启动 WebUI

Colab 无法直接访问本地端口。若想用 Web 界面，可用隧道工具（如 cloudflared / ngrok）把 `127.0.0.1:7860` 暴露成公网链接：

```python
# 后台启动 WebUI（默认 IndexTTS-2.5）
!nohup uv run webui.py --host 0.0.0.0 --port 7860 > webui.log 2>&1 &
!sleep 20 && tail -20 webui.log
```

```python
# 用 cloudflared 生成公网链接（无需注册）
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://127.0.0.1:7860
```